# 🔥 Heavy CPU Benchmark Notebook (100% CPU Spike on Workers)
This notebook distributes 300 Million trigonometric and floating-point math calculations across 64 partitions to saturate **100% CPU Utilization** across all 3 Worker machines.

### Instructions:
1. Run the code cell below.
2. Open Task Manager or `htop` on worker machines (`ahmed-worker`, `mohamostafa`, `desktop-v7dh9ca`).
3. Observe **100% CPU Spike** across all 32 Logical Processors!

In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Initialize PySpark Session connected to Master
spark = SparkSession.builder \
    .appName("Jupyter-Heavy-CPU-Benchmark") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.cores", "4") \
    .getOrCreate()

start_time = time.time()

# 2. Generate 3 Million rows and repartition into 64 partitions
print("Generating dataset and repartitioning to 64 partitions...")
df_raw = spark.range(0, 3000000) \
    .withColumn("passenger_count", (F.col("id") % 6) + 1) \
    .withColumn("trip_distance", F.round((F.col("id") % 20) + 0.5, 2)) \
    .withColumn("fare_amount", F.round((F.col("id") % 50) + 2.5, 2)) \
    .withColumn("total_amount", F.col("fare_amount") + 5.0)

df_dist = df_raw.repartition(64)

print("🔥 LAUNCHING 300 MILLION MATH CALCULATIONS (WATCH TASK MANAGER ON ALL WORKERS)...")
heavy_df = df_dist.withColumn("heavy_calc", F.expr(
    "transform(sequence(1, 100), x -> sin(x * trip_distance) + cos(x * fare_amount) + atan(x * total_amount))"
)).withColumn("calc_sum", F.expr("aggregate(heavy_calc, 0D, (acc, x) -> acc + x)"))

result = heavy_df.groupBy("passenger_count").agg(
    F.count("*").alias("total_trips"),
    F.round(F.avg("calc_sum"), 4).alias("avg_heavy_score")
).orderBy("passenger_count")

result.show(20, truncate=False)

elapsed = time.time() - start_time
print(f"✅ Heavy CPU Benchmark Completed in {elapsed:.2f} seconds!")

ModuleNotFoundError: No module named 'pyspark'